In [5]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col
from pyspark.sql.functions import max, avg, min
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import when



In [27]:
spark = SparkSession.builder.master("local").appName("Test").getOrCreate()
print(spark)


In [17]:
spark = SparkSession.builder.master("local").appName("Test").getOrCreate()
df = spark.read.csv("C:/Users/egor2/trainee_de/task4_data/ds_salaries.csv")
                   
df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)



In [19]:
df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)



In [29]:
schema = StructType([
    StructField("index", IntegerType(), True),
    StructField("work_year", IntegerType(), True),
    StructField("experience_level", StringType(), True),
    StructField("employment_type", StringType(), True),
    StructField("job_title", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("salary_currency", StringType(), True),
    StructField("salary_in_usd", IntegerType(), True),
    StructField("employee_residence", StringType(), True),
    StructField("remote_ratio", IntegerType(), True),
    StructField("company_location", StringType(), True),
    StructField("company_size", StringType(), True)
])

csv_path = "C:/Users/egor2/trainee_de/task4_data/ds_salaries.csv"  
df = spark.read.csv(csv_path, header=True, schema=schema, inferSchema=False)


df.printSchema()

df.show()



root
 |-- index: integer (nullable = true)
 |-- work_year: integer (nullable = true)
 |-- experience_level: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- salary_currency: string (nullable = true)
 |-- salary_in_usd: integer (nullable = true)
 |-- employee_residence: string (nullable = true)
 |-- remote_ratio: integer (nullable = true)
 |-- company_location: string (nullable = true)
 |-- company_size: string (nullable = true)

+-----+---------+----------------+---------------+--------------------+--------+---------------+-------------+------------------+------------+----------------+------------+
|index|work_year|experience_level|employment_type|           job_title|  salary|salary_currency|salary_in_usd|employee_residence|remote_ratio|company_location|company_size|
+-----+---------+----------------+---------------+--------------------+--------+---------------+-------------+---

In [31]:
display(df.toPandas())

,index,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L
...,...,...,...,...,...,...,...,...,...,...,...,...
602,602,2022,SE,FT,Data Engineer,154000,USD,154000,US,100,US,M
603,603,2022,SE,FT,Data Engineer,126000,USD,126000,US,100,US,M
604,604,2022,SE,FT,Data Analyst,129000,USD,129000,US,0,US,M
605,605,2022,SE,FT,Data Analyst,150000,USD,150000,US,100,US,M


In [38]:
df_job_title = df.select("job_title").distinct()

df_job_title.printSchema()

df_job_title.show()


root
 |-- job_title: string (nullable = true)

+--------------------+
|           job_title|
+--------------------+
|3D Computer Visio...|
|  Lead Data Engineer|
|Head of Machine L...|
|     Data Specialist|
| Data Analytics Lead|
|Machine Learning ...|
|   Lead Data Analyst|
|Data Engineering ...|
|Staff Data Scientist|
|       ETL Developer|
|Director of Data ...|
|Product Data Analyst|
|Principal Data Sc...|
|        AI Scientist|
|Director of Data ...|
|Machine Learning ...|
| Lead Data Scientist|
|Machine Learning ...|
|Data Science Engi...|
|Machine Learning ...|
+--------------------+
only showing top 20 rows



In [40]:
df_job_title = df.select("job_title").distinct()

df_job_title.printSchema()

df_job_title.show(truncate=False)


root
 |-- job_title: string (nullable = true)

+----------------------------------------+
|job_title                               |
+----------------------------------------+
|3D Computer Vision Researcher           |
|Lead Data Engineer                      |
|Head of Machine Learning                |
|Data Specialist                         |
|Data Analytics Lead                     |
|Machine Learning Scientist              |
|Lead Data Analyst                       |
|Data Engineering Manager                |
|Staff Data Scientist                    |
|ETL Developer                           |
|Director of Data Engineering            |
|Product Data Analyst                    |
|Principal Data Scientist                |
|AI Scientist                            |
|Director of Data Science                |
|Machine Learning Engineer               |
|Lead Data Scientist                     |
|Machine Learning Infrastructure Engineer|
|Data Science Engineer                   |
|Machin

In [42]:
df_analytic = df.groupBy("job_title").agg(
    max("salary_in_usd").alias("max_salary"),
    avg("salary_in_usd").alias("avg_salary"),
    min("salary_in_usd").alias("min_salary")
)

df_analytic.printSchema()

df_analytic.show()


root
 |-- job_title: string (nullable = true)
 |-- max_salary: integer (nullable = true)
 |-- avg_salary: double (nullable = true)
 |-- min_salary: integer (nullable = true)

+--------------------+----------+------------------+----------+
|           job_title|max_salary|        avg_salary|min_salary|
+--------------------+----------+------------------+----------+
|3D Computer Visio...|      5409|            5409.0|      5409|
|  Lead Data Engineer|    276000|          139724.5|     56000|
|Head of Machine L...|     79039|           79039.0|     79039|
|     Data Specialist|    165000|          165000.0|    165000|
| Data Analytics Lead|    405000|          405000.0|    405000|
|Machine Learning ...|    260000|          158412.5|     12000|
|   Lead Data Analyst|    170000|           92203.0|     19609|
|Data Engineering ...|    174000|          123227.2|     59303|
|Staff Data Scientist|    105000|          105000.0|    105000|
|       ETL Developer|     54957|           54957.0|     

In [44]:
df_analytic = df.groupBy("job_title").agg(
    max("salary_in_usd").alias("max_salary"),
    avg("salary_in_usd").alias("avg_salary"),
    min("salary_in_usd").alias("min_salary")
)

df_analytic.printSchema()

df_analytic.show(truncate=False)

root
 |-- job_title: string (nullable = true)
 |-- max_salary: integer (nullable = true)
 |-- avg_salary: double (nullable = true)
 |-- min_salary: integer (nullable = true)

+----------------------------------------+----------+------------------+----------+
|job_title                               |max_salary|avg_salary        |min_salary|
+----------------------------------------+----------+------------------+----------+
|3D Computer Vision Researcher           |5409      |5409.0            |5409      |
|Lead Data Engineer                      |276000    |139724.5          |56000     |
|Head of Machine Learning                |79039     |79039.0           |79039     |
|Data Specialist                         |165000    |165000.0          |165000    |
|Data Analytics Lead                     |405000    |405000.0          |405000    |
|Machine Learning Scientist              |260000    |158412.5          |12000     |
|Lead Data Analyst                       |170000    |92203.0         